# Group 5420: how much does shuffled CV overstate next-day stock AUC?

Main project notebook. Run the cells from top to bottom.

**Before the first run:** in VS Code, click **Select Kernel** (top right) and choose **Python (finalpaper)**.

Where the code lives: the functions are in `src/` (for example `src/features.py`). This notebook calls them and shows the results.

In [1]:
import sys
import pandas as pd

from src.config import SEED, DATA_CSV
from src.load_data import load_prices, check_prices
from src.features import FEATURES, build_model_table

print("Python:", sys.executable)   # should end in envs/finalpaper/bin/python
print("SEED =", SEED)

Python: /opt/anaconda3/envs/finalpaper/bin/python
SEED = 0


## Step 0: load the raw prices and check them
Expected: 514,075 rows, 100 tickers, no duplicates, no non-positive prices, no missing values.

In [2]:
raw = load_prices()      # reads data/NASDAQ100_Historical_Data.csv, sorted by Ticker and Date
checks = check_prices(raw)

Total rows:        514,075
Tickers:           100
Date range:        2000-01-03 to 2026-02-18
Duplicate (Ticker, Date) rows: 0
Rows with a non-positive price: 0
Missing values:    0



Per-ticker start / end dates (sorted by start date):
             start         end  rows
Ticker                              
AAPL    2000-01-03  2026-02-18  6571
ADBE    2000-01-03  2026-02-18  6571
ADI     2000-01-03  2026-02-18  6571
ADP     2000-01-03  2026-02-18  6571
ADSK    2000-01-03  2026-02-18  6571
AEP     2000-01-03  2026-02-18  6571
AMAT    2000-01-03  2026-02-18  6571
AMD     2000-01-03  2026-02-18  6571
AMGN    2000-01-03  2026-02-18  6571
AMZN    2000-01-03  2026-02-18  6571
ASML    2000-01-03  2026-02-18  6571
BIIB    2000-01-03  2026-02-18  6571
BKNG    2000-01-03  2026-02-18  6571
BKR     2000-01-03  2026-02-18  6571
CCEP    2000-01-03  2026-02-18  6571
CDNS    2000-01-03  2026-02-18  6571
CMCSA   2000-01-03  2026-02-18  6571
COST    2000-01-03  2026-02-18  6571
CPRT    2000-01-03  2026-02-18  6571
CSGP    2000-01-03  2026-02-18  6571
CTAS    2000-01-03  2026-02-18  6571
CTSH    2000-01-03  2026-02-18  6571
DLTR    2000-01-03  2026-02-18  6571
EA      2000-01-03  2

## Step 1: label and 23 features (Qidi's `make_features`, in `src/features.py`)
Expected: 491,772 rows, 23 features + target, about 50.58% up days.

In [3]:
table = build_model_table(raw)
print(f"{len(FEATURES)} features, {len(table):,} rows, {100 * table['target'].mean():.2f}% up days")
table.head()

23 features, 491,772 rows, 50.58% up days


,Ticker,Date,return_1d,return_5d,return_10d,return_20d,sma_ratio_5,sma_ratio_10,sma_ratio_20,sma_ratio_50,...,volatility_10d,volatility_20d,volatility_60d,volume_sma_ratio_10,volume_sma_ratio_20,daily_range,avg_range_10d,high_low_ratio,atr_14,target
199,AAPL,2000-10-16,-0.030303,-0.030303,-0.111111,-0.648352,1.032258,0.993789,0.608944,0.455840,...,0.958468,1.978783,1.271015,0.763957,0.671539,0.105263,0.093531,1.105263,0.166713,0.0
200,AAPL,2000-10-17,-0.062500,-0.032258,-0.090909,-0.666667,0.974026,0.940439,0.605449,0.432526,...,0.918769,1.977105,1.265593,0.647046,0.486102,0.111111,0.092142,1.114286,0.173062,0.0
201,AAPL,2000-10-18,0.000000,0.034483,-0.142857,-0.670330,0.967742,0.955414,0.645161,0.437573,...,0.838964,1.973469,1.263342,0.962496,0.657839,0.138889,0.098888,1.151515,0.170621,0.0
202,AAPL,2000-10-19,-0.066667,-0.066667,-0.151515,-0.670588,0.915033,0.906149,0.641466,0.413589,...,0.853888,1.973582,1.268274,1.619882,1.143188,0.058824,0.091950,1.060606,0.176953,1.0
203,AAPL,2000-10-20,0.035714,-0.121212,-0.121212,-0.628205,0.973154,0.950820,0.703883,0.433742,...,0.889545,1.987192,1.267555,0.834661,0.598883,0.057143,0.090164,1.058824,0.163151,1.0


## Step 2: look-ahead test (professor's requirement)

The rule: the features for day t may only use prices up to day t.

The test: pick a day t, multiply every price and volume **after** t by a random number between 0.5 and 2, then recompute all features. Every feature on or before t must stay **exactly** the same.

### 2a. Look at one example by hand
Cut-off day t = 2011-12-30. We compare the features before and after scrambling the future.

In [4]:
from src.features import features_by_ticker
from tests.test_no_lookahead import scramble_after   # the same scramble the formal test uses

cutoff = "2011-12-30"
original = features_by_ticker(raw)[FEATURES]
changed_raw, future = scramble_after(raw, cutoff, SEED)
recomputed = features_by_ticker(changed_raw)[FEATURES]

diff = (recomputed - original).abs()
print("Rows on or before t:", (~future).sum(), "| largest change in any feature:", diff[~future].max().max())
print("Rows after t:       ", future.sum(), "| largest change in any feature:", diff[future].max().max())

Rows on or before t: 196104 | largest change in any feature: 0.0
Rows after t:        317971 | largest change in any feature: 60.33531016042596


In [5]:
# AAPL around the cut-off day: return_1d is identical up to 2011-12-30, then differs
aapl = (raw["Ticker"] == "AAPL") & raw["Date"].between("2011-12-27", "2012-01-05")
pd.DataFrame({
    "Date": raw.loc[aapl, "Date"].dt.date,
    "price (original)": raw.loc[aapl, "Adj Close"],
    "price (scrambled)": changed_raw.loc[aapl, "Adj Close"].round(2),
    "return_1d (original)": original.loc[aapl, "return_1d"].round(5),
    "return_1d (scrambled)": recomputed.loc[aapl, "return_1d"].round(5),
})

,Date,price (original),price (scrambled),return_1d (original),return_1d (scrambled)
3015,2011-12-27,12.18,12.18,0.00828,0.00828
3016,2011-12-28,12.06,12.06,-0.00985,-0.00985
3017,2011-12-29,12.14,12.14,0.00663,0.00663
3018,2011-12-30,12.14,12.14,0.00000,0.00000
3019,2012-01-03,12.32,23.21,0.01483,0.91193
3020,2012-01-04,12.39,21.90,0.00568,-0.05641
3021,2012-01-05,12.53,15.71,0.01130,-0.28266


### 2b. Run the formal test (4 cut-off days + a check that the test can catch a leaky feature)
All 5 lines should say **PASSED**. Rerun this cell after ANY change to `src/features.py`.

In [6]:
!{sys.executable} -m pytest -v tests/test_no_lookahead.py

============================= test session starts ==============================
platform darwin -- Python 3.11.18, pytest-9.2.0, pluggy-1.7.0 -- /opt/anaconda3/envs/finalpaper/bin/python
cachedir: .pytest_cache
rootdir: /Users/wangjin/Desktop/26Fall/ESE542/542_finalpaper
configfile: pytest.ini
plugins: platformdirs-4.14.5, anyio-4.16.0
collecting ... 

collected 5 items                                                              

tests/test_no_lookahead.py::test_features_ignore_future_prices[2005-06-30] 

PASSED [ 20%]
tests/test_no_lookahead.py::test_features_ignore_future_prices[2011-12-30] 

PASSED [ 40%]
tests/test_no_lookahead.py::test_features_ignore_future_prices[2018-03-15] 

PASSED [ 60%]
tests/test_no_lookahead.py::test_features_ignore_future_prices[2023-12-29] 

PASSED [ 80%]
tests/test_no_lookahead.py::test_check_catches_a_leaky_feature 

PASSED    [100%]

============================== 5 passed in 3.92s ===============================
